In [ ]:
import sys
sys.path.insert(0, "../../run")
from run_config import REPO_PATH, SEASONS, DATA_PROCESSING_PATH
sys.path.insert(1, f"{REPO_PATH}")

import pandas as pd
import joblib

In [6]:
processed_data_path='../../data/processed/premier_league/'

In [7]:
seasons=sorted(SEASONS)

In [8]:
data_dfs=[pd.read_csv(f"{processed_data_path}/{season}/all_data_df.csv") for season in seasons]

# Home-Away features

In [ ]:
import pandas as pd

class TeamLagFeatureGenerator:
    def __init__(self, lookback=5, date_col='date', home_col='home', away_col='away'):
        self.lookback = lookback
        self.date_col = date_col
        self.home_col = home_col
        self.away_col = away_col

    def transform(self, season_dfs):
        all_rows = []

        for season_df in season_dfs:
            df = season_df.copy()
            df[self.date_col] = pd.to_datetime(df[self.date_col])
            df = df.sort_values(self.date_col)

            # Track match histories for each team
            team_history = {}

            new_rows = []

            for idx, row in df.iterrows():
                home_team = row[self.home_col]
                away_team = row[self.away_col]
                match_date = row[self.date_col]

                lag_row = row.copy()

                for team_role, team_name in [(self.home_col, home_team), (self.away_col, away_team)]:
                    history = team_history.get(team_name, [])

                    for lag_i in range(1, self.lookback + 1):
                        if len(history) >= lag_i:
                            hist_entry = history[-lag_i]
                            hist_row = hist_entry['row']
                            was_home = hist_entry['was_home']

                            # Copy all numerical columns except identifiers
                            for col in df.columns:
                                if col not in [self.date_col, self.home_col, self.away_col]:
                                    lag_row[f'{team_role}_lag{lag_i}_{col}'] = hist_row[col]

                            # Add one indicator only per lag
                            lag_row[f'{team_role}_lag{lag_i}_was_home'] = int(was_home)
                        else:
                            for col in df.columns:
                                if col not in [self.date_col, self.home_col, self.away_col]:
                                    lag_row[f'{team_role}_lag{lag_i}_{col}'] = None
                            lag_row[f'{team_role}_lag{lag_i}_was_home'] = None

                new_rows.append(lag_row)

                # Store current match in team history
                team_history.setdefault(home_team, []).append({'row': row, 'was_home': True})
                team_history.setdefault(away_team, []).append({'row': row, 'was_home': False})

            all_rows.extend(new_rows)

        return pd.DataFrame(all_rows)


In [27]:
generator = TeamLagFeatureGenerator(lookback=5)
result_df = generator.transform(data_dfs)

In [29]:
result_df

,home_performance_pk,home_performance_pkatt,home_performance_crdr,home_performance_touches,home_performance_tkl,home_performance_int,home_performance_blocks,home_expected_xg,home_expected_npxg,home_expected_xag,...,away_lag5_away_performance_int.1,away_lag5_away_performance_tklw,away_lag5_away_performance_pkwon,away_lag5_away_performance_pkcon,away_lag5_away_performance_og,away_lag5_away_performance_recov,away_lag5_away_aerial_duels_won,away_lag5_away_aerial_duels_lost,away_lag5_away_aerial_duels_won%,away_lag5_was_home
234,0,0,0,620,14,7,12,1.1,1.1,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,0,0,0,638,11,10,9,0.1,0.1,0.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
155,0,0,0,400,15,12,7,1.1,1.1,1.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,2,2,0,612,13,14,9,2.7,1.2,0.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,0,0,0,695,9,4,12,1.1,1.1,1.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,0,0,0,657,25,11,7,2.7,2.7,2.6,...,10.0,11.0,2.0,0.0,0.0,40.0,5.0,8.0,38.5,1.0
321,0,0,0,694,19,7,6,1.1,1.1,0.6,...,1.0,6.0,0.0,0.0,0.0,42.0,24.0,19.0,55.8,1.0
206,0,0,0,613,27,7,13,2.2,2.2,1.9,...,12.0,3.0,0.0,0.0,0.0,38.0,12.0,11.0,52.2,1.0
17,0,0,0,446,22,6,15,1.7,1.7,1.6,...,8.0,10.0,0.0,0.0,1.0,43.0,12.0,7.0,63.2,1.0


In [32]:
pd.concat(data_dfs[1:]).shape

(1510, 217)